<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-09-ai-agents/lesson-9.6-computer-use-agents/notebooks/exercises-9_6.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 9.6: Computer Use Agents

*Module 9 · AI Agents & Autonomous Systems*

> Function-calling agents interact with APIs. Computer use agents interact with screens. They see screenshots, move mice, click buttons, type text — controlling a computer like a human would. Anthropic pioneered this with Claude’s Computer Use API. This lesson covers three approaches: API-based browser automation, vision-based screen control, and Anthropic’s native computer use — all with critical safety constraints.

`Computer Use` · `Playwright` · `Vision Agents` · `Safety` · `60 min`

---

## About this notebook

This notebook contains the **3 runnable code examples** from the published lesson `lesson-9.6.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Browser Automation with Playwright + LLM — `01_playwright_agent.py`
2. Step 2 — Anthropic Computer Use — Native Screen Control — `02_anthropic_computer_use.py`
3. Step 3 — Gemini Vision Agent — Screenshot-Based Control — `03_vision_agent.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q anthropic>=0.40.0 google-generativeai pillow


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export ANTHROPIC_API_KEY=sk-...
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("ANTHROPIC_API_KEY", "YOUR_ANTHROPIC_API_KEY_HERE")
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Browser Automation with Playwright + LLM

The LLM decides what to do. Playwright executes browser actions.

**`01_playwright_agent.py`** · _Block 1 of 3_

BROWSER AGENT — Playwright + Gemini — pip install playwright google-generativeai


In [ ]:
# BROWSER AGENT — Playwright + Gemini
# pip install playwright google-generativeai
# playwright install chromium
import google.generativeai as genai
from playwright.sync_api import sync_playwright
import base64, json, os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class BrowserAgent:
    """LLM-driven browser: screenshot → decide → act."""
    ALLOWED_DOMAINS = ["example.com", "wikipedia.org"]  # Safety: allowlist
    MAX_ACTIONS = 10  # Safety: action cap

    def __init__(self):
        self.model = genai.GenerativeModel("gemini-2.0-flash")
        self.actions_taken = 0

    def _is_safe_url(self, url):
        return any(d in url for d in self.ALLOWED_DOMAINS)

    def run(self, task):
        with sync_playwright() as p:
            browser = p.chromium.launch(headless=True)
            page = browser.new_page(viewport={"width":1280,"height":720})
            history = []

            for step in range(self.MAX_ACTIONS):
                # 1. Screenshot (observe)
                screenshot = page.screenshot()
                img_b64 = base64.b64encode(screenshot).decode()

                # 2. Ask LLM what to do (think)
                prompt = (f"Task: {task}\nHistory: {history[-3:]}\n"
                          f"Current URL: {page.url}\n"
                          f"What action next? Respond with JSON:\n"
                          f'{{"action":"goto|click|type|done","selector":"css","value":"text","url":"..."}}')
                resp = self.model.generate_content([
                    {"mime_type":"image/png", "data":img_b64}, prompt
                ])
                raw = resp.text.strip()
                if raw.startswith("```"): raw = raw.split("\n",1)[1].rsplit("```",1)[0]
                try: action = json.loads(raw)
                except: action = {"action":"done"}

                # 3. Execute action (act) with safety checks
                act = action.get("action")
                if act == "done":
                    return {"result": page.content()[:500], "steps": step+1}
                elif act == "goto":
                    url = action.get("url","")
                    if self._is_safe_url(url):
                        page.goto(url); history.append(f"goto {url}")
                    else: history.append(f"BLOCKED: {url} not in allowlist")
                elif act == "click":
                    page.click(action.get("selector",""))
                    history.append(f"clicked {action.get('selector')}")
                elif act == "type":
                    page.fill(action.get("selector",""), action.get("value",""))
                    history.append(f"typed into {action.get('selector')}")

            return {"result": "Max actions reached", "steps": self.MAX_ACTIONS}

print("Playwright Browser Agent:\n")
print("  Loop: screenshot → LLM decides action → Playwright executes")
print("  Safety: domain allowlist, max 10 actions, headless VM")
print("  Actions: goto, click, type, done")


> **What just happened?** The browser agent loop: take a screenshot → send to Gemini with the task → Gemini returns a JSON action (goto/click/type/done) → Playwright executes it → repeat. Three safety layers: domain allowlist blocks unauthorized URLs, action cap prevents infinite loops, headless mode prevents screen takeover.


### Step 2 · Anthropic Computer Use — Native Screen Control

Claude sees screenshots, returns pixel coordinates. The most capable computer use API.

**`02_anthropic_computer_use.py`** · _Block 2 of 3_

ANTHROPIC COMPUTER USE — Claude controls the screen — pip install anthropic


In [ ]:
# ANTHROPIC COMPUTER USE — Claude controls the screen
# pip install anthropic
import anthropic, base64, json, os

class ClaudeComputerAgent:
    """Anthropic Computer Use: screenshot → Claude → coordinates."""
    def __init__(self):
        self.client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
        self.actions_log = []

    def step(self, screenshot_b64, task, history=""):
        """Send screenshot + task to Claude, get action back."""
        resp = self.client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=1024,
            tools=[{
                "type": "computer_20250124",
                "name": "computer",
                "display_width_px": 1280,
                "display_height_px": 720,
                "display_number": 1,
            }],
            messages=[{
                "role": "user",
                "content": [
                    {"type":"image", "source":{"type":"base64",
                        "media_type":"image/png", "data":screenshot_b64}},
                    {"type":"text", "text":f"Task: {task}\nHistory: {history}\nWhat action next?"}
                ]
            }]
        )
        # Parse Claude's response for tool_use actions
        for block in resp.content:
            if block.type == "tool_use" and block.name == "computer":
                action = block.input  # {action: "click", coordinate: [640, 360]}
                self.actions_log.append(action)
                return action
        return {"action": "done"}

print("Anthropic Computer Use:\n")
print("  Tool type: computer_20250124")
print("  Claude sees screenshot, returns actions:")
print("    click(coordinate=[x,y])")
print("    type(text='hello')")
print("    key(key='Enter')")
print("    scroll(coordinate=[x,y], direction='down')")
print("    screenshot() — request a new screenshot")
print("  Resolution: must match display_width_px/height_px")
print("  ALWAYS run in sandbox (Docker/VM). Never on host machine.")


### Step 3 · Gemini Vision Agent — Screenshot-Based Control

Same pattern with Gemini: screenshot + task → structured action output.

**`03_vision_agent.py`** · _Block 3 of 3_

GEMINI VISION AGENT — Screen control via screenshots


In [ ]:
# GEMINI VISION AGENT — Screen control via screenshots
import google.generativeai as genai
from PIL import Image, ImageDraw
import json, os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class VisionControlAgent:
    """See screen → decide action → return coordinates."""
    def __init__(self, safety_rules=None):
        self.model = genai.GenerativeModel("gemini-2.0-flash")
        self.rules = safety_rules or [
            "Never click \'Delete\' or \'Remove\' buttons",
            "Never fill in payment or password fields",
            "Never navigate away from allowed domains",
        ]

    def decide(self, screenshot, task, history=""):
        rules_text = "\n".join(f"  - {r}" for r in self.rules)
        prompt = (
            f"You control a computer screen. Task: {task}\n"
            f"History: {history}\n\n"
            f"SAFETY RULES (never violate):\n{rules_text}\n\n"
            f"Look at the screenshot. What action next?\n"
            f"Respond JSON: {{\"action\":\"click|type|scroll|done\","
            f"\"x\":int,\"y\":int,\"text\":\"for typing\",\"reason\":\"why\"}}")
        resp = self.model.generate_content([screenshot, prompt])
        raw = resp.text.strip()
        if raw.startswith("```"): raw = raw.split("\n",1)[1].rsplit("```",1)[0]
        try: return json.loads(raw)
        except: return {"action":"done","reason":"parse error"}

# Demo with synthetic screenshot
img = Image.new("RGB",(800,400),"white")
d = ImageDraw.Draw(img)
d.rectangle([50,50,200,80], fill="#0284c7"); d.text((80,55),"Search",fill="white")
d.rectangle([50,100,400,130], outline="gray"); d.text((60,105),"Enter query...",fill="gray")
d.text((50,160),"GenAI Course - 14999 INR",fill="black")
d.text((50,190),"Python Course - 9999 INR",fill="black")

agent = VisionControlAgent()
action = agent.decide(img, "Find the GenAI course price")
print(f"Vision Agent Decision:")
print(f"  Action: {action.get('action')}")
print(f"  Position: ({action.get('x',0)}, {action.get('y',0)})")
print(f"  Reason: {action.get('reason','')}")


> **What just happened?** Safety is non-negotiable for computer use agents. A single misclick can send an email, make a purchase, or delete data. Defense in depth: sandbox + allowlists + HITL + credential isolation.


---

## Wrap-up

You've walked through all 3 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
